# 01 - Data Ingestion

Pull the two data sources used throughout this project:
- **Yahoo Finance daily OHLCV** for BTC-USD (multi-year macro backbone, spans all four labeled events).
- **Binance's free public tick/kline archive** (`data.binance.vision`, no API key) for real trade-by-trade data and 1-minute klines, pulled only for the reference period and windows around each event (Yahoo has no tick data at any lookback -- it only ever stores pre-built bars).

In [1]:
import sys
sys.path.insert(0, '..')
import yaml
import pandas as pd

cfg = yaml.safe_load(open('../configs/horizons.yaml'))
cfg

{'asset': 'BTCUSDT',
 'yahoo_ticker': 'BTC-USD',
 'horizons': {'micro': {'description': 'Tick-level microstructure, ~minutes',
   'source': 'binance_ticks',
   'window': 1000,
   'step': 500,
   'depth': 3,
   'channels': ['time', 'log_price', 'signed_volume']},
  'intraday': {'description': 'Hourly bars, ~1 day - 1 week',
   'source': 'binance_klines_1m',
   'resample': '1h',
   'window': 48,
   'step': 24,
   'depth': 4,
   'channels': ['time', 'log_return', 'volume']},
  'macro': {'description': 'Daily bars, ~1-3 months',
   'source': 'yahoo_daily',
   'window': 30,
   'step': 10,
   'depth': 4,
   'channels': ['time', 'log_return', 'volume']}},
 'reference_period': {'start': '2023-10-01', 'end': '2023-12-15'},
 'events': {'covid_crash': {'start': '2020-02-20', 'end': '2020-04-30'},
  'meme_stock_era_btc_runup_crash': {'start': '2021-01-01',
   'end': '2021-05-31'},
  'crypto_crash_terra_luna': {'start': '2022-05-01', 'end': '2022-05-31'},
  'crypto_crash_ftx': {'start': '2022-11-01

## Yahoo daily (macro horizon)

In [2]:
from src.ingestion.yahoo import fetch_daily

daily = fetch_daily(cfg['yahoo_ticker'])
if daily.index.tz is None:
    daily.index = daily.index.tz_localize('UTC')
print(daily.shape, daily.index.min(), daily.index.max())
daily.tail()

(4369, 5) 2014-09-17 00:00:00+00:00 2026-09-03 00:00:00+00:00


Price,open,high,low,close,volume
timestamp,,,,,
2026-08-29 00:00:00+00:00,77830.890625,78327.945312,77381.898438,78245.812500,14053131399
2026-08-30 00:00:00+00:00,78246.171875,79373.179688,77056.148438,77667.570312,19314839672
2026-08-31 00:00:00+00:00,77673.703125,79247.343750,77378.437500,78548.632812,30608386987
2026-09-01 00:00:00+00:00,78539.859375,79196.515625,76399.062500,77403.625000,30608796575
2026-09-03 00:00:00+00:00,77310.171875,77845.062500,76993.968750,77597.007812,26850246656


## Binance klines (intraday horizon) and reference-period ticks (micro horizon)

Klines are pulled for the full history needed by the intraday horizon's construction; raw trade ticks are pulled only for the reference period plus each event window, per the project plan (a full multi-year tick archive is far more data than a class project needs).

In [3]:
from src.ingestion.binance import fetch_klines, fetch_trades

symbol = cfg['asset']
ref = cfg['reference_period']

klines_ref = fetch_klines(symbol, '1m', ref['start'], ref['end'])
print('reference klines:', klines_ref.shape)

trades_ref = fetch_trades(symbol, ref['start'], ref['end'])
print('reference trades:', trades_ref.shape)

reference klines: (109440, 5)


reference trades: (98826819, 6)


In [4]:
# Pull klines + a few days of ticks around each labeled event (tick data is
# large -- keep the tick pull to a short window right around each event start).
event_data = {}
for name, ev in cfg['events'].items():
    k = fetch_klines(symbol, '1m', ev['start'], ev['end'])
    event_data[name] = {'klines': k}
    print(name, 'klines:', k.shape)

covid_crash klines: (101962, 5)


meme_stock_era_btc_runup_crash klines: (216837, 5)


crypto_crash_terra_luna klines: (44640, 5)


crypto_crash_ftx klines: (43200, 5)


banking_crisis_2023 klines: (44560, 5)


Tick data for specific event days can be pulled on demand in notebook 05 (`fetch_trades(symbol, event_start, event_start_plus_a_few_days)`) -- for the micro horizon's event study, only look at a handful of days around each event's start rather than the full event window, to keep download size manageable.